## Mixed-Precision

In [8]:
import torch
torch.set_printoptions(precision=60)
torch.tensor(1/3, dtype=torch.float64)

tensor(0.333333333333333314829616256247390992939472198486328125000000,
       dtype=torch.float64)

In [9]:
torch.tensor(1/3, dtype=torch.float32)

tensor(0.333333343267440795898437500000000000000000000000000000000000)

In [10]:
torch.tensor(1/3, dtype=torch.float16)

tensor(0.333251953125000000000000000000000000000000000000000000000000,
       dtype=torch.float16)

In [12]:
torch.__version__

'2.13.0'

## Numerical Precision-type

In [11]:
torch.finfo(torch.float32)

finfo(resolution=1e-06, min=-3.40282e+38, max=3.40282e+38, eps=1.19209e-07, smallest_normal=1.17549e-38, tiny=1.17549e-38, dtype=float32)

In [13]:
torch.finfo(torch.float16)

finfo(resolution=0.001, min=-65504, max=65504, eps=0.000976562, smallest_normal=6.10352e-05, tiny=6.10352e-05, dtype=float16)

In [14]:
torch.finfo(torch.float64)

finfo(resolution=1e-15, min=-1.79769e+308, max=1.79769e+308, eps=2.22045e-16, smallest_normal=2.22507e-308, tiny=2.22507e-308, dtype=float64)

In [19]:
num = 3.40282e+38
print(f"{num:.0f}")

340282000000000014192072600942972764160


## A Mixed-Precision Code Example

Finetuning Benchmarks

* Large Movie Review Dataset: https://ai.stanford.edu/~amaas/data/sentiment/

In [2]:
## Load dataset

import os
import sys
import tarfile
import time

import numpy as np
import pandas as pd
from packaging import version
from torch.utils.data import Dataset
from tqdm import tqdm
import urllib


def reporthook(count, block_size, total_size):
    global start_time
    if count == 0:
        start_time = time.time()
        return
    duration = time.time() - start_time
    progress_size = int(count * block_size)
    speed = progress_size / (1024.0**2 * duration)
    percent = count * block_size * 100.0 / total_size

    sys.stdout.write(
        f"\r{int(percent)}% | {progress_size / (1024.**2):.2f} MB "
        f"| {speed:.2f} MB/s | {duration:.2f} sec elapsed"
    )
    sys.stdout.flush()


def download_dataset():
    source = "http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
    target = "aclImdb_v1.tar.gz"

    if os.path.exists(target):
        os.remove(target)

    if not os.path.isdir("aclImdb") and not os.path.isfile("aclImdb_v1.tar.gz"):
        urllib.request.urlretrieve(source, target, reporthook)

    if not os.path.isdir("aclImdb"):

        with tarfile.open(target, "r:gz") as tar:
            tar.extractall()


def load_dataset_into_to_dataframe():
    basepath = "aclImdb"

    labels = {"pos": 1, "neg": 0}

    df = pd.DataFrame()

    with tqdm(total=50000) as pbar:
        for s in ("test", "train"):
            for l in ("pos", "neg"):
                path = os.path.join(basepath, s, l)
                for file in sorted(os.listdir(path)):
                    with open(os.path.join(path, file), "r", encoding="utf-8") as infile:
                        txt = infile.read()

                    if version.parse(pd.__version__) >= version.parse("1.3.2"):
                        x = pd.DataFrame(
                            [[txt, labels[l]]], columns=["review", "sentiment"]
                        )
                        df = pd.concat([df, x], ignore_index=False)

                    else:
                        df = df.append([[txt, labels[l]]], ignore_index=True)
                    pbar.update()
    df.columns = ["text", "label"]

    np.random.seed(0)
    df = df.reindex(np.random.permutation(df.index))

    print("Class distribution:")
    np.bincount(df["label"].values)

    return df


def partition_dataset(df):
    df_shuffled = df.sample(frac=1, random_state=1).reset_index()

    df_train = df_shuffled.iloc[:35_000]
    df_val = df_shuffled.iloc[35_000:40_000]
    df_test = df_shuffled.iloc[40_000:]

    df_train.to_csv("train.csv", index=False, encoding="utf-8")
    df_val.to_csv("val.csv", index=False, encoding="utf-8")
    df_test.to_csv("test.csv", index=False, encoding="utf-8")


class IMDBDataset(Dataset):
    def __init__(self, dataset_dict, partition_key="train"):
        self.partition = dataset_dict[partition_key]

    def __getitem__(self, index):
        return self.partition[index]

    def __len__(self):
        return self.partition.num_rows

- Float32 regular high

In [1]:
import os
import os.path as op
import time

from datasets import load_dataset
from lightning import Fabric
import torch
from torch.utils.data import DataLoader
import torchmetrics
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from watermark import watermark

/Users/user/Projects/basillians.github.io/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

def tokenize_text(batch):
    return tokenizer(batch["text"], truncation=True, padding=True)


def train(num_epochs, model, optimizer, train_loader, val_loader, fabric):

    for epoch in range(num_epochs):
        train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=2).to(fabric.device)

        model.train()
        for batch_idx, batch in enumerate(train_loader):
            model.train()


            ### FORWARD AND BACK PROP   
            outputs = model(batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["label"]) 
            optimizer.zero_grad()
            
            # For non-Fabric PyTorch:
            #outputs["loss"].backward()
            fabric.backward(outputs["loss"])

            ### UPDATE MODEL PARAMETERS
            optimizer.step()

            ### LOGGING
            if not batch_idx % 300:
                print(f"Epoch: {epoch+1:04d}/{num_epochs:04d} | Batch {batch_idx:04d}/{len(train_loader):04d} | Loss: {outputs['loss']:.4f}")

            model.eval()
            with torch.no_grad():
                predicted_labels = torch.argmax(outputs["logits"], 1)
                train_acc.update(predicted_labels, batch["label"])

        ### MORE LOGGING
        model.eval()
        with torch.no_grad():
            val_acc = torchmetrics.Accuracy(task="multiclass", num_classes=2).to(fabric.device)
            for batch in val_loader:

                outputs = model(batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["label"])
                predicted_labels = torch.argmax(outputs["logits"], 1)
                val_acc.update(predicted_labels, batch["label"])

            print(f"Epoch: {epoch+1:04d}/{num_epochs:04d} | Train acc.: {train_acc.compute()*100:.2f}% | Val acc.: {val_acc.compute()*100:.2f}%")
            train_acc.reset(), val_acc.reset()


if __name__ == "__main__":

    print(watermark(packages="torch,lightning,transformers", python=True))
    print("Torch CUDA available?", torch.cuda.is_available())
    
    # Dynamically select the best available hardware accelerator
    if torch.cuda.is_available():
        device = torch.device("cuda")          # For NVIDIA GPUs
    elif torch.backends.mps.is_available():
        device = torch.device("mps")           # For Apple Silicon (M1/M2/M3/M4)
    else:
        device = torch.device("cpu")           # Fallback to CPU

    print(f"Using device: {device}")

    # torch.set_float32_matmul_precision("high")
    torch.manual_seed(123)

    ##########################
    ### 1 Loading the Dataset
    ##########################
    download_dataset()
    df = load_dataset_into_to_dataframe()
    if not (op.exists("train.csv") and op.exists("val.csv") and op.exists("test.csv")):
        partition_dataset(df)

    imdb_dataset = load_dataset(
        "csv",
        data_files={
            "train": "train.csv",
            "validation": "val.csv",
            "test": "test.csv",
        },
    )

    #########################################
    ### 2 Tokenization and Numericalization
    #########################################

    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    print("Tokenizer input max length:", tokenizer.model_max_length, flush=True)
    print("Tokenizer vocabulary size:", tokenizer.vocab_size, flush=True)

    print("Tokenizing ...", flush=True)
    imdb_tokenized = imdb_dataset.map(tokenize_text, batched=True, batch_size=None)
    del imdb_dataset
    imdb_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    #########################################
    ### 3 Set Up DataLoaders
    #########################################

    train_dataset = IMDBDataset(imdb_tokenized, partition_key="train")
    val_dataset = IMDBDataset(imdb_tokenized, partition_key="validation")
    test_dataset = IMDBDataset(imdb_tokenized, partition_key="test")

    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=12,
        shuffle=True, 
        num_workers=0,
        drop_last=True,
    )

    val_loader = DataLoader(
        dataset=val_dataset,
        batch_size=12,
        num_workers=0,
        drop_last=True,
    )

    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=12,
        num_workers=0,
        drop_last=True,
    )


    #########################################
    ### 4 Initializing the Model
    #########################################

    fabric = Fabric(accelerator="mps", devices=1, precision="32-true")
    fabric.launch()

    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=2)

    # model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

    model, optimizer = fabric.setup(model, optimizer)
    train_loader, val_loader, test_loader = fabric.setup_dataloaders(train_loader, val_loader, test_loader)
    fabric.barrier()

    #########################################
    ### 5 Finetuning
    #########################################

    start = time.time()
    train(
        num_epochs=3,
        model=model,
        optimizer=optimizer,
        train_loader=train_loader,
        val_loader=val_loader,
        fabric=fabric
    )

    end = time.time()
    elapsed = end-start
    print(f"Time elapsed {elapsed/60:.2f} min")

    with torch.no_grad():
        model.eval()
        test_acc = torchmetrics.Accuracy(task="multiclass", num_classes=2).to(fabric.device)
        for batch in test_loader:

            outputs = model(batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["label"])
            predicted_labels = torch.argmax(outputs["logits"], 1)
            test_acc.update(predicted_labels, batch["label"])

    # print(f"Memory used: {torch.cuda.max_memory_reserved() / 1e9:.02f} GB")
    if torch.backends.mps.is_available():
        # Current allocated memory by PyTorch tensors on MPS
        mem_allocated = torch.mps.current_allocated_memory() / 1e9
        # Memory allocated by the Metal driver (includes some overhead)
        mem_driver = torch.mps.driver_allocated_memory() / 1e9
        print(f"MPS memory allocated: {mem_allocated:.2f} GB")
        print(f"MPS driver allocated: {mem_driver:.2f} GB")
    else:
        print(f"CUDA memory used: {torch.cuda.max_memory_reserved() / 1e9:.2f} GB")
    print(f"Test accuracy {test_acc.compute()*100:.2f}%")

Python implementation: CPython
Python version       : 3.11.14
IPython version      : 9.16.1

torch       : 2.13.0
lightning   : 2.6.5
transformers: 5.15.1

Torch CUDA available? False
Using device: mps


100%|██████████| 50000/50000 [17:26<00:00, 47.76it/s] 


Class distribution:
Tokenizer input max length: 512
Tokenizer vocabulary size: 30522
Tokenizing ...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 5486.62it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch: 0001/0003 | Batch 0000/2916 | Loss: 0.7009
Epoch: 0001/0003 | Batch 0300/2916 | Loss: 0.3120
Epoch: 0001/0003 | Batch 0600/2916 | Loss: 0.3955
Epoch: 0001/0003 | Batch 0900/2916 | Loss: 0.2393
Epoch: 0001/0003 | Batch 1200/2916 | Loss: 0.3225
Epoch: 0001/0003 | Batch 1500/2916 | Loss: 0.2625
Epoch: 0001/0003 | Batch 1800/2916 | Loss: 0.1362
Epoch: 0001/0003 | Batch 2100/2916 | Loss: 0.1507
Epoch: 0001/0003 | Batch 2400/2916 | Loss: 0.4005
Epoch: 0001/0003 | Batch 2700/2916 | Loss: 0.1408
Epoch: 0001/0003 | Train acc.: 89.37% | Val acc.: 91.77%
Epoch: 0002/0003 | Batch 0000/2916 | Loss: 0.3071
Epoch: 0002/0003 | Batch 0300/2916 | Loss: 0.0483
Epoch: 0002/0003 | Batch 0600/2916 | Loss: 0.0760
Epoch: 0002/0003 | Batch 0900/2916 | Loss: 0.0822
Epoch: 0002/0003 | Batch 1200/2916 | Loss: 0.0060
Epoch: 0002/0003 | Batch 1500/2916 | Loss: 0.0700
Epoch: 0002/0003 | Batch 1800/2916 | Loss: 0.0087
Epoch: 0002/0003 | Batch 2100/2916 | Loss: 0.3403
Epoch: 0002/0003 | Batch 2400/2916 | Loss: 

- 16-mixed

In [4]:
def tokenize_text(batch):
    return tokenizer(batch["text"], truncation=True, padding=True)


def train(num_epochs, model, optimizer, train_loader, val_loader, fabric):

    for epoch in range(num_epochs):
        train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=2).to(fabric.device)

        model.train()
        for batch_idx, batch in enumerate(train_loader):
            model.train()


            ### FORWARD AND BACK PROP   
            outputs = model(batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["label"]) 
            optimizer.zero_grad()
            
            # For non-Fabric PyTorch:
            #outputs["loss"].backward()
            fabric.backward(outputs["loss"])

            ### UPDATE MODEL PARAMETERS
            optimizer.step()

            ### LOGGING
            if not batch_idx % 300:
                print(f"Epoch: {epoch+1:04d}/{num_epochs:04d} | Batch {batch_idx:04d}/{len(train_loader):04d} | Loss: {outputs['loss']:.4f}")

            model.eval()
            with torch.no_grad():
                predicted_labels = torch.argmax(outputs["logits"], 1)
                train_acc.update(predicted_labels, batch["label"])

        ### MORE LOGGING
        model.eval()
        with torch.no_grad():
            val_acc = torchmetrics.Accuracy(task="multiclass", num_classes=2).to(fabric.device)
            for batch in val_loader:

                outputs = model(batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["label"])
                predicted_labels = torch.argmax(outputs["logits"], 1)
                val_acc.update(predicted_labels, batch["label"])

            print(f"Epoch: {epoch+1:04d}/{num_epochs:04d} | Train acc.: {train_acc.compute()*100:.2f}% | Val acc.: {val_acc.compute()*100:.2f}%")
            train_acc.reset(), val_acc.reset()


if __name__ == "__main__":

    print(watermark(packages="torch,lightning,transformers", python=True))
    print("Torch CUDA available?", torch.cuda.is_available())
    
    # Dynamically select the best available hardware accelerator
    if torch.cuda.is_available():
        device = torch.device("cuda")          # For NVIDIA GPUs
    elif torch.backends.mps.is_available():
        device = torch.device("mps")           # For Apple Silicon (M1/M2/M3/M4)
    else:
        device = torch.device("cpu")           # Fallback to CPU

    print(f"Using device: {device}")

    # torch.set_float32_matmul_precision("high")
    torch.manual_seed(123)

    ##########################
    ### 1 Loading the Dataset
    ##########################
    download_dataset()
    df = load_dataset_into_to_dataframe()
    if not (op.exists("train.csv") and op.exists("val.csv") and op.exists("test.csv")):
        partition_dataset(df)

    imdb_dataset = load_dataset(
        "csv",
        data_files={
            "train": "train.csv",
            "validation": "val.csv",
            "test": "test.csv",
        },
    )

    #########################################
    ### 2 Tokenization and Numericalization
    #########################################

    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    print("Tokenizer input max length:", tokenizer.model_max_length, flush=True)
    print("Tokenizer vocabulary size:", tokenizer.vocab_size, flush=True)

    print("Tokenizing ...", flush=True)
    imdb_tokenized = imdb_dataset.map(tokenize_text, batched=True, batch_size=None)
    del imdb_dataset
    imdb_tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    os.environ["TOKENIZERS_PARALLELISM"] = "false"

    #########################################
    ### 3 Set Up DataLoaders
    #########################################

    train_dataset = IMDBDataset(imdb_tokenized, partition_key="train")
    val_dataset = IMDBDataset(imdb_tokenized, partition_key="validation")
    test_dataset = IMDBDataset(imdb_tokenized, partition_key="test")

    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=12,
        shuffle=True, 
        num_workers=0,
        drop_last=True,
    )

    val_loader = DataLoader(
        dataset=val_dataset,
        batch_size=12,
        num_workers=0,
        drop_last=True,
    )

    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=12,
        num_workers=0,
        drop_last=True,
    )


    #########################################
    ### 4 Initializing the Model
    #########################################

    fabric = Fabric(accelerator="mps", devices=1, precision="16-mixed")
    fabric.launch()

    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=2)

    # model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)

    model, optimizer = fabric.setup(model, optimizer)
    train_loader, val_loader, test_loader = fabric.setup_dataloaders(train_loader, val_loader, test_loader)
    fabric.barrier()

    #########################################
    ### 5 Finetuning
    #########################################

    start = time.time()
    train(
        num_epochs=3,
        model=model,
        optimizer=optimizer,
        train_loader=train_loader,
        val_loader=val_loader,
        fabric=fabric
    )

    end = time.time()
    elapsed = end-start
    print(f"Time elapsed {elapsed/60:.2f} min")

    with torch.no_grad():
        model.eval()
        test_acc = torchmetrics.Accuracy(task="multiclass", num_classes=2).to(fabric.device)
        for batch in test_loader:

            outputs = model(batch["input_ids"], attention_mask=batch["attention_mask"], labels=batch["label"])
            predicted_labels = torch.argmax(outputs["logits"], 1)
            test_acc.update(predicted_labels, batch["label"])

    # print(f"Memory used: {torch.cuda.max_memory_reserved() / 1e9:.02f} GB")
    if torch.backends.mps.is_available():
        # Current allocated memory by PyTorch tensors on MPS
        mem_allocated = torch.mps.current_allocated_memory() / 1e9
        # Memory allocated by the Metal driver (includes some overhead)
        mem_driver = torch.mps.driver_allocated_memory() / 1e9
        print(f"MPS memory allocated: {mem_allocated:.2f} GB")
        print(f"MPS driver allocated: {mem_driver:.2f} GB")
    else:
        print(f"CUDA memory used: {torch.cuda.max_memory_reserved() / 1e9:.2f} GB")
    print(f"Test accuracy {test_acc.compute()*100:.2f}%")

Python implementation: CPython
Python version       : 3.11.14
IPython version      : 9.16.1

torch       : 2.13.0
lightning   : 2.6.5
transformers: 5.15.1

Torch CUDA available? False
Using device: mps


100%|██████████| 50000/50000 [17:45<00:00, 46.92it/s] 


Class distribution:
Tokenizer input max length: 512
Tokenizer vocabulary size: 30522
Tokenizing ...


Using 16-bit Automatic Mixed Precision (AMP)
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4793.38it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch: 0001/0003 | Batch 0000/2916 | Loss: 0.7008
Epoch: 0001/0003 | Batch 0300/2916 | Loss: 0.3654
Epoch: 0001/0003 | Batch 0600/2916 | Loss: 0.4304
Epoch: 0001/0003 | Batch 0900/2916 | Loss: 0.3136
Epoch: 0001/0003 | Batch 1200/2916 | Loss: 0.3135
Epoch: 0001/0003 | Batch 1500/2916 | Loss: 0.1414
Epoch: 0001/0003 | Batch 1800/2916 | Loss: 0.0735
Epoch: 0001/0003 | Batch 2100/2916 | Loss: 0.0880
Epoch: 0001/0003 | Batch 2400/2916 | Loss: 0.4565
Epoch: 0001/0003 | Batch 2700/2916 | Loss: 0.0539
Epoch: 0001/0003 | Train acc.: 89.60% | Val acc.: 92.19%
Epoch: 0002/0003 | Batch 0000/2916 | Loss: 0.1191
Epoch: 0002/0003 | Batch 0300/2916 | Loss: 0.0106
Epoch: 0002/0003 | Batch 0600/2916 | Loss: 0.0317
Epoch: 0002/0003 | Batch 0900/2916 | Loss: 0.1566
Epoch: 0002/0003 | Batch 1200/2916 | Loss: 0.0057
Epoch: 0002/0003 | Batch 1500/2916 | Loss: 0.0555
Epoch: 0002/0003 | Batch 1800/2916 | Loss: 0.0399
Epoch: 0002/0003 | Batch 2100/2916 | Loss: 0.1174
Epoch: 0002/0003 | Batch 2400/2916 | Loss: 